## Table of Contents
### 1. [Import Libraries](#import-libraries)

In [76]:
# Packages for data manipulation
import pandas as pd
import numpy as np

# Packages for data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# APIs for data access
# import kaggle
# from kaggle.api.kaggle_api_extended import KaggleApi

# Packages for machine learning
import sklearn as sk
from sklearn.model_selection import train_test_split

# Packages for data preprocessing
from sklearn.preprocessing import  RobustScaler

### 2. [Load & Inspect Data](#load--inspect-data)

In [77]:
df = pd.read_csv('./data_from_kaggle/train.csv')


### 3. [Data Cleaning](#Data-Cleaning)

* Drop Duplicates

In [78]:
# Any duplicate rows were found in the dataset.
df[df.duplicated()]

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice


* Handling Missing Data

In [79]:
# 1. Fill absence indicators 
none_cols = ['PoolQC','MiscFeature','Alley','Fence','FireplaceQu',
             'GarageType','GarageFinish','GarageQual','GarageCond',
             'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1',
             'BsmtFinType2','MasVnrType']
df[none_cols] = df[none_cols].fillna('None')

# 2. LotFrontage: fill missing values with median by neighborhood
lotfrontage_median = df.groupby('Neighborhood')['LotFrontage'].transform('median')
df['LotFrontage'] = df['LotFrontage'].fillna(lotfrontage_median)

# 3. GarageYrBlt: fill with median year
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(df['GarageYrBlt'].median())

# 4. MasVnrArea: fill with 0
df['MasVnrArea'] = df['MasVnrArea'].fillna(0)

# 5. Electrical: fill with mode
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

In [80]:
df.isnull().sum().sort_values(ascending=False)

Id             0
CentralAir     0
GarageYrBlt    0
GarageType     0
FireplaceQu    0
              ..
MasVnrArea     0
MasVnrType     0
Exterior2nd    0
Exterior1st    0
SalePrice      0
Length: 81, dtype: int64

* Creating Custom Features

In [81]:
df["SQR_RATIO"] = df["GrLivArea"] /  df["LotArea"]
df["SQR_ROOM"] = df["GrLivArea"] / df["TotRmsAbvGrd"]
df["TOTAL_ROOMS"] = df["TotRmsAbvGrd"] + df["FullBath"] + df["HalfBath"] + df["BsmtFullBath"] + df["BsmtHalfBath"]
df["TOTAL_AREA"] = df["GrLivArea"] + df["TotalBsmtSF"]
df["TOTAL_VOL"]= (df["MasVnrArea"] * np.sqrt(df["GrLivArea"])) / 4

* Encoding Categorical Variables

In [82]:

label_encoder = LabelEncoder()
# Encode categorical features

categorical_features = df.select_dtypes(include=['object'])

for col in categorical_features.columns:
    df[col] = label_encoder.fit_transform(df[col].astype(str))

# Checking if label encoding was successful
df.head(2)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,MoSold,YrSold,SaleType,SaleCondition,SalePrice,SQR_RATIO,SQR_ROOM,TOTAL_ROOMS,TOTAL_AREA,TOTAL_VOL
0,1,60,3,65.0,8450,1,1,3,3,0,...,2,2008,8,4,208500,0.202367,213.750000,12,2566,2026.255167
1,2,20,3,80.0,9600,1,1,3,3,0,...,5,2007,8,4,181500,0.131458,210.333333,9,2524,0.000000


* Correlation Analysis

In [83]:
correlations = df.corr()
print(correlations['SalePrice'].apply(abs).sort_values(ascending = False).iloc[:60])

selected_columns = []

SalePrice        1.000000
OverallQual      0.790982
TOTAL_AREA       0.778959
GrLivArea        0.708624
TOTAL_ROOMS      0.644795
GarageCars       0.640409
ExterQual        0.636884
GarageArea       0.623431
TotalBsmtSF      0.613581
1stFlrSF         0.605852
BsmtQual         0.593734
KitchenQual      0.589189
FullBath         0.560664
SQR_ROOM         0.540645
TotRmsAbvGrd     0.533723
YearBuilt        0.522897
TOTAL_VOL        0.519832
YearRemodAdd     0.507101
MasVnrArea       0.472614
Fireplaces       0.466929
GarageYrBlt      0.466754
GarageFinish     0.425684
GarageType       0.415283
HeatingQC        0.400178
BsmtFinSF1       0.386420
Foundation       0.382479
LotFrontage      0.349876
WoodDeckSF       0.324413
2ndFlrSF         0.319334
OpenPorchSF      0.315856
BsmtExposure     0.309043
HalfBath         0.284108
LotArea          0.263843
LotShape         0.255580
CentralAir       0.251328
GarageCond       0.246705
Electrical       0.234945
PavedDrive       0.231357
BsmtFullBath

In [84]:
for col in df.columns:
    if abs(correlations[col]['SalePrice']) > 0.10:
        selected_columns.append(col)

df_selected = df[selected_columns]
df_selected.head(2)

,MSZoning,LotFrontage,LotArea,LotShape,Neighborhood,HouseStyle,OverallQual,YearBuilt,YearRemodAdd,RoofStyle,...,EnclosedPorch,ScreenPorch,PoolQC,Fence,SaleCondition,SalePrice,SQR_ROOM,TOTAL_ROOMS,TOTAL_AREA,TOTAL_VOL
0,3,65.0,8450,3,5,5,7,2003,2003,1,...,0,0,3,4,4,208500,213.750000,12,2566,2026.255167
1,3,80.0,9600,3,24,2,6,1976,1976,1,...,0,0,3,4,4,181500,210.333333,9,2524,0.000000


* Feature Transformation (Scaling, Normalization)

In [85]:
scaler = RobustScaler()
scaled_array = scaler.fit_transform(df_selected) 

df_scaled = pd.DataFrame(scaled_array, columns=df_selected.columns)
df_scaled.head(2)

,MSZoning,LotFrontage,LotArea,LotShape,Neighborhood,HouseStyle,OverallQual,YearBuilt,YearRemodAdd,RoofStyle,...,EnclosedPorch,ScreenPorch,PoolQC,Fence,SaleCondition,SalePrice,SQR_ROOM,TOTAL_ROOMS,TOTAL_AREA,TOTAL_VOL
0,0.0,-0.25,-0.254076,0.0,-0.7,1.0,0.5,0.652174,0.243243,0.0,...,0.0,0.0,0.0,0.0,0.0,0.541506,-0.223265,1.0,0.087481,1.34694
1,0.0,0.50,0.030015,0.0,1.2,0.0,0.0,0.065217,-0.486486,0.0,...,0.0,0.0,0.0,0.0,0.0,0.220173,-0.283094,0.0,0.045249,0.00000


In [86]:
# Dropping the target variable from the features.

df_scaled.drop(columns=['SalePrice'], inplace=True)
df_scaled.shape


(1460, 56)

In [87]:
df_scaled.to_csv('./default.csv', index=False)

### 3. [Modeling / Statistical Analysis](#modeling--statistical-analysis) 

In [88]:
default = pd.read_csv('./default.csv')

   - [Data train/test split](#Data-train-/-test-split)

In [89]:
random_state = 42
x = default
y = df['SalePrice']

In [90]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=random_state)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((1314, 56), (146, 56), (1314,), (146,))

   - [XGBoost](#XGBoost)

In [91]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
from typing import Tuple, Literal, Optional
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score



In [100]:

xgb_base = XGBRegressor(
    objective="reg:squarederror",
    n_jobs=-1,
    random_state=42
)

param_dist = {
    "n_estimators": randint(360, 1200),
    "learning_rate": uniform(0.01, 0.2),   # 0.01 ~ 0.21
    "max_depth": randint(3, 15),
    "min_child_weight": randint(1,8),
    "subsample": uniform(0.6, 0.4),        # 0.6 ~ 1.0
    "colsample_bytree": uniform(0.6, 0.4), # 0.6 ~ 1.0
    "gamma": uniform(0.0, 0.5),
    "reg_alpha": uniform(0.0, 0.1),        # L1
    "reg_lambda": uniform(0.7, 0.6)        # L2 (0.7 ~ 1.3)
}

search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=50,            
    scoring="neg_root_mean_squared_error",
    cv=4,
    random_state=80,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)
best_model = search.best_estimator_
print("Best Params:", search.best_params_)

y_pred = best_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f"[Tuned XGB] R²: {r2:.4f} | MAE: {mae:.2f} | MSE: {mse:.2f} | RMSE: {rmse:.2f}")



Fitting 4 folds for each of 50 candidates, totalling 200 fits
Best Params: {'colsample_bytree': 0.6505404250816037, 'gamma': 0.1711063515812276, 'learning_rate': 0.0258036032396442, 'max_depth': 4, 'min_child_weight': 3, 'n_estimators': 862, 'reg_alpha': 0.0016966687010190108, 'reg_lambda': 1.1633016575424624, 'subsample': 0.7571133679815375}
[Tuned XGB] R²: 0.9224 | MAE: 15332.08 | MSE: 708777344.00 | RMSE: 26622.87


* Train/Test Split : 60/40

[Tuned XGB] R²: 0.8825 | MAE: 17080.12 | MSE: 851658304.00 | RMSE: 29183.19

[Upsampled]R²: 0.8898 | MAE: 16238.55 | MSE: 798810176.00 | RMSE: 28263.23

* Train/Test Split : 70/30

[Tuned XGB] R²: 0.9158 | MAE: 15754.56 | MSE: 587408192.00 | RMSE: 24236.51

[Upsampled]R²: 0.9177 | MAE: 15442.79 | MSE: 574444800.00 | RMSE: 23967.58

* Train/Test Split : 80/20

[Tuned XGB] R²: 0.9035 | MAE: 16520.29 | MSE: 740106944.00 | RMSE: 27204.9

[Upsampled]R²: 0.9126 | MAE: 16160.11 | MSE: 670441152.00 | RMSE: 25892.88

* Train/Test Split : 90/10

[Tuned XGB] R²: 0.9224 | MAE: 15332.08 | MSE: 708777344.00 | RMSE: 26622.87

[Upsampled]R²: 0.9206 | MAE: 15638.94 | MSE: 725179712.00 | RMSE: 26929.16 

   - [XGBoost-up-sampling](#XGBoost-up-sampling)

In [94]:
def upsample_by_bins(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    q: int = 10,
    target_mode: Literal["max", "quantile"] = "max",
    target_quantile: float = 0.8,
    random_state: int = 42,
    shuffle: bool = True,
) -> Tuple[pd.DataFrame, pd.Series, pd.Series, pd.Series]:

    df = X_train.copy()
    df["_y_"] = y_train.values

    # 依分位數分箱（duplicates="drop" 可避免邊界重疊錯誤）
    df["_bin_"] = pd.qcut(df["_y_"], q=q, labels=False, duplicates="drop")

    bin_counts_before = df["_bin_"].value_counts().sort_index()

    # 決定補到的目標數量
    if target_mode == "max":
        target_n = int(bin_counts_before.max())
    elif target_mode == "quantile":
        target_n = int(bin_counts_before.quantile(target_quantile))
        target_n = max(target_n, int(bin_counts_before.median()))  # 基本保護，避免太小
    else:
        raise ValueError('target_mode must be "max" or "quantile".')

    parts = []
    for b, grp in df.groupby("_bin_"):
        if len(grp) < target_n:
            need = target_n - len(grp)
            up = grp.sample(n=need, replace=True, random_state=random_state)
            grp = pd.concat([grp, up], axis=0)
        parts.append(grp)

    df_up = pd.concat(parts, axis=0)
    if shuffle:
        df_up = df_up.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

    bin_counts_after = df_up["_bin_"].value_counts().sort_index()

    X_up = df_up.drop(columns=["_y_", "_bin_"])
    y_up = df_up["_y_"]

    return X_up, y_up, bin_counts_before, bin_counts_after


In [95]:
def print_metrics(y_true, y_pred, prefix=""):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"{prefix}R²: {r2:.4f} | MAE: {mae:.2f} | MSE: {mse:.2f} | RMSE: {rmse:.2f}")



In [96]:
# 1) 先做上採樣（只動訓練集）
X_train_up, y_train_up, before, after = upsample_by_bins(
    X_train, y_train,
    q=10,                 # 分箱數（可試 8/10/12）
    target_mode="max",    # 或 "quantile"
    target_quantile=0.8,  # 若 target_mode="quantile" 時才用到
    random_state=42
)

print("Bin counts BEFORE:\n", before)
print("Bin counts AFTER:\n",  after)

# 2) 重新訓練模型（先用你調好的最佳參數；或用同一組 baseline）
from xgboost import XGBRegressor

xgb_up = XGBRegressor(
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    objective="reg:squarederror"
)

xgb_up.fit(X_train_up, y_train_up)
y_pred_up = xgb_up.predict(X_test)

print_metrics(y_test, y_pred_up, prefix="[Upsampled]")


Bin counts BEFORE:
 _bin_
0    132
1    132
2    131
3    135
4    128
5    130
6    132
7    135
8    127
9    132
Name: count, dtype: int64
Bin counts AFTER:
 _bin_
0    135
1    135
2    135
3    135
4    135
5    135
6    135
7    135
8    135
9    135
Name: count, dtype: int64
[Upsampled]R²: 0.9206 | MAE: 15638.94 | MSE: 725179712.00 | RMSE: 26929.16
